In [0]:
# ---------------------------------------------------------------------------
# NETWORK REACHABILITY TEST
# Checks whether this Databricks workspace can reach the websites the
# pipeline needs. Many workspaces block outbound internet by default, so
# run this before relying on any live downloads.
#
# How to read the output:
#   REACHED 200  -> the site answered normally; the pipeline can use it
#   REACHED 403  -> the site answered but refused us (e.g. blocks scripts/bots)
#   REACHED 404  -> the site answered but the page doesn't exist (check the URL)
#   BLOCKED      -> no answer at all: the workspace's network, a firewall,
#                   DNS or a timeout stopped the request before it got there
#
# If GitHub is REACHED but the NPS sites are BLOCKED, the workspace has
# internet access but those specific sites are being blocked. If everything
# is BLOCKED, the workspace has no outbound internet access at all.
# ---------------------------------------------------------------------------

import requests   # library for making web requests (preinstalled on Databricks)


# --- Sites to test: a readable name for each, mapped to its URL ---
sites = {
    # The Blue Ridge Parkway road closures page: the source the pipeline scrapes
    "NPS road status page": "https://www.nps.gov/blri/planyourvisit/roadclosures.htm",

    # NPS's ArcGIS map service for parkway mileposts (location data for each milepost)
    "NPS ArcGIS mileposts": "https://services1.arcgis.com/fBc8EJBxQRMcHlei/arcgis/rest/services/SEAC_BLRI_Mileposts/FeatureServer",

    # Control test: a well-known site unrelated to the project. If this works
    # but the NPS sites don't, the problem is specific to those sites.
    "GitHub": "https://raw.githubusercontent.com/github/gitignore/main/Python.gitignore",
}


# --- Try each site and print the result ---
for name, url in sites.items():
    try:
        # Request the page. timeout=15 gives up after 15 seconds, so a
        # blocked site fails quickly instead of hanging the notebook.
        response = requests.get(url, timeout=15)

        # Any reply, even an error page, means the site was reachable.
        # The status code says whether the reply was actually usable (see above).
        print(f"REACHED  {response.status_code}  {name}")

    except Exception as error:
        # No reply at all. The error type explains why, for example:
        #   ConnectionError -> network or firewall blocked it, or DNS failed
        #   Timeout         -> no answer within 15 seconds
        #   SSLError        -> a secure connection couldn't be set up
        print(f"BLOCKED  {name}  ({type(error).__name__}: {error})")